# Data processing: raw files to one monthly table

This notebook turns the seven raw NOAA/GLOS files in `data/raw/` into
`data/processed/monthly.csv`, the single table every later step uses. The raw files are
only read, never changed.

Nothing is defined here. The reading and merging functions live in
`function_declaration.ipynb`; the next cell loads them. Run this notebook before
`main.ipynb`.

In [1]:
%run ./function_declaration.ipynb

## 1. Precipitation and evaporation

**Intuition.** These four files (two lakes, two variables) have one row per year and
twelve monthly columns. A model needs time running down the rows, so `read_depth_table`
reshapes each file into one long series with one value per month, in mm.

In [2]:
precip_mic = read_depth_table("prc_mic_lake.csv")
precip_hur = read_depth_table("prc_hur_lake.csv")
evap_mic = read_depth_table("evaporation_mic.csv")
evap_hur = read_depth_table("evaporation_hur.csv")

print(f"{len(precip_mic)} months, {precip_mic.index.min()} to {precip_mic.index.max()}")
pd.DataFrame({"precip_mic": precip_mic, "precip_hur": precip_hur,
              "evap_mic": evap_mic, "evap_hur": evap_hur}).loc["1950-01":"1950-06"]

1020 months, 1940-01 to 2024-12


,precip_mic,precip_hur,evap_mic,evap_hur
1950-01,76.71,80.15,103.478,86.145
1950-02,49.56,74.62,53.968,55.733
1950-03,58.92,58.01,46.870,44.194
1950-04,86.08,80.49,28.923,21.278
1950-05,49.85,34.55,-0.472,-2.713
1950-06,85.40,72.75,-2.002,-4.275


**A known problem in the early years.** Lake Michigan evaporation for 1941-1944 is nearly
the same from one year to the next, which real weather never is. It looks filled in with
modelled or average values. The table below puts those years side by side with 1950-1953
for comparison. This is why the analysis starts in 1950.

In [3]:
by_year = evap_mic.groupby([evap_mic.index.year, evap_mic.index.month]).first().unstack()
by_year.loc[[1941, 1942, 1943, 1944, 1950, 1951, 1952, 1953]].round(1)

,1,2,3,4,5,6,7,8,9,10,11,12
1941,90.4,44.8,28.2,8.7,-0.9,1.8,15.5,37.8,58.7,66.0,101.3,113.7
1942,97.4,46.1,31.1,9.1,-0.4,2.7,17.0,38.5,59.0,66.3,101.8,114.2
1943,98.5,46.3,31.4,9.2,-0.3,2.9,17.2,38.6,59.1,66.4,101.9,114.3
1944,98.6,47.6,30.6,8.6,-0.4,2.9,17.9,39.4,60.3,67.0,103.7,113.8
1950,103.5,54.0,46.9,28.9,-0.5,-2.0,8.2,46.5,44.5,49.2,136.5,122.2
1951,62.4,38.2,36.4,1.8,-3.2,-4.9,-1.8,22.2,67.8,52.8,114.5,93.7
1952,52.2,33.3,34.6,10.7,-1.1,-3.7,10.2,34.2,69.4,124.5,79.6,79.0
1953,74.5,44.2,30.3,17.4,-0.7,-0.5,21.9,34.9,90.7,60.0,88.6,125.4


## 2. Runoff

**Intuition.** Runoff is the river water draining from the land into the lake. It is
already one row per month, but it is a *flow* in m³/s, and it comes with
`fraction_gauged`: the share of the basin that actually had river gauges that month (the
rest is estimated). `NaN` marks months with no data.

In [4]:
runoff_mic = read_runoff("runoff_mic_arm.csv")
runoff_hur = read_runoff("runoff_hur_arm.csv")

for name, ro in [("Michigan", runoff_mic), ("Huron", runoff_hur)]:
    have = ro["flow_cms"].dropna()
    print(f"{name}: flow available {have.index.min()} to {have.index.max()}")
runoff_mic.loc["2024-05":"2024-12"]

Michigan: flow available 1901-03 to 2024-08
Huron: flow available 1901-04 to 2024-08


,flow_cms,fraction_gauged
YYYY-MM,,
2024-05,1712.0,0.979
2024-06,1424.0,0.979
2024-07,1340.0,0.979
2024-08,854.0,0.979
2024-09,NaN,0.783
2024-10,NaN,0.000
2024-11,NaN,0.000
2024-12,NaN,0.000


**Turning a flow into a depth.** Precipitation and evaporation are depths (mm over the
lake), so runoff has to be put in the same unit before the three can be added. Multiply
the flow by the seconds in the month to get a volume, then spread that volume over the
lake surface. The two basins are added first, because flows add.

In [5]:
flow_total = runoff_mic["flow_cms"] + runoff_hur["flow_cms"]
runoff_mm = runoff_to_depth_mm(flow_total)
pd.DataFrame({"flow_cms": flow_total, "runoff_mm": runoff_mm}).loc["1950-01":"1950-06"].round(1)

,flow_cms,runoff_mm
YYYY-MM,,
1950-01,2131.0,48.6
1950-02,1515.0,31.2
1950-03,3001.0,68.5
1950-04,4323.0,95.4
1950-05,2098.0,47.9
1950-06,1317.0,29.1


## 3. Lake level

**Intuition.** The level file is labelled "Huron". Michigan and Huron are one lake
hydraulically, so it should be the combined level, but that is worth checking. The
record low and record high of Lake Michigan-Huron are well known (January 2013 and
October 1986), so see whether this series has its extremes in the same months.

In [6]:
level = read_levels()
have = level.dropna()
print(f"level values: {have.index.min()} to {have.index.max()} ({level.isna().sum()} empty months in the file)")
print(f"lowest:  {have.min():.2f} m in {have.idxmin()}")
print(f"highest: {have.max():.2f} m in {have.idxmax()}")

level values: 1918-01 to 2023-12 (22 empty months in the file)
lowest:  175.57 m in 2013-01
highest: 177.50 m in 1986-10


## 4. Merge everything into one table

**Intuition.** `build_monthly` repeats the steps above and joins them:

- precipitation and evaporation are depths, so the two lakes are combined as an
  area-weighted average (Michigan 57,800 km², Huron 59,600 km²);
- runoff is added across the two basins and converted to mm;
- `nbs_mm` = precipitation − evaporation + runoff, the net basin supply;
- `dlevel_mm` is the change in level from the previous month, the quantity the models predict;
- rows before 1950 and after the last month with every series present are dropped.

In [7]:
monthly = build_monthly()
print(f"{len(monthly)} months, {monthly.index.min()} to {monthly.index.max()}")
monthly.head()

888 months, 1950-01 to 2023-12


,level_m,precip_mm,evap_mm,runoff_mm,frac_gauged_mic,frac_gauged_hur,nbs_mm,dlevel_mm
month,,,,,,,,
1950-01,175.96,78.456371,94.678624,48.617295,0.778,0.743,32.395043,20.0
1950-02,175.99,62.282112,54.864031,31.218807,0.778,0.743,38.636889,30.0
1950-03,176.02,58.458024,45.511486,68.465744,0.778,0.743,81.412283,30.0
1950-04,176.17,83.242147,25.041893,95.444770,0.778,0.743,153.645024,150.0
1950-05,176.27,42.082709,-1.609680,47.864422,0.778,0.743,91.556811,100.0


## 5. Checks

**No gaps.** Every month between the first and last should be present, with no missing values.

In [8]:
expected = pd.period_range(monthly.index.min(), monthly.index.max(), freq="M")
print("every month present:", monthly.index.equals(expected))
print("missing values:     ", int(monthly.isna().sum().sum()))

every month present: True
missing values:      0


**Reasonable ranges.** A quick look at each column's spread.

In [9]:
monthly.describe().T[["mean", "std", "min", "max"]].round(2)

,mean,std,min,max
level_m,176.51,0.42,175.57,177.50
precip_mm,70.94,28.71,12.64,233.43
evap_mm,43.50,36.15,-6.27,142.54
runoff_mm,37.24,19.13,12.21,115.73
frac_gauged_mic,0.94,0.05,0.78,0.98
frac_gauged_hur,0.86,0.05,0.74,0.93
nbs_mm,64.68,58.31,-90.71,279.87
dlevel_mm,0.56,68.13,-160.00,260.00


**Runoff coverage.** The gauged share of each basin changes over time. It is kept in the
table as a data-quality column and is not used as a predictor.

In [10]:
monthly[["frac_gauged_mic", "frac_gauged_hur"]].agg(["min", "max"]).round(2)

,frac_gauged_mic,frac_gauged_hur
min,0.78,0.74
max,0.98,0.93


## 6. Save

In [11]:
out = PROCESSED / "monthly.csv"
monthly.to_csv(out, float_format="%.4f")
print(f"wrote {out.relative_to(ROOT)}")

wrote data/processed/monthly.csv
